# Semana 11: Matriz de Consistencia y Plan de Generalización
Este notebook documenta la versión preliminar de la matriz de consistencia y el plan de generalización para el proyecto:
**Agente Inteligente para la Generación Automática de Queries MongoDB a partir de Lenguaje Natural**.
Incluye la estructura solicitada y evidencia de los indicadores clave.

## Plan para Generalización fuera de la muestra (Plan B)
- **Generalización propuesta:** El sistema será validado con ejemplos reales de Prosegur, pero se diseñará para aceptar instrucciones en lenguaje natural de otros dominios y colecciones, permitiendo su adaptación a nuevos contextos mediante la expansión del diccionario de sinónimos y reglas contextuales.
- **Plan B (si no se puede generalizar):** Si la generalización no es viable por limitaciones de vocabulario o estructura, se documentará el proceso de adaptación y se propondrá un módulo de configuración para que los analistas puedan personalizar el sistema según nuevos esquemas o colecciones.

## Ejemplo de matriz de consistencia en formato tabla (actualizada y relacionada con los resultados obtenidos en el cuaderno)
| Problema | Objetivo general/específicos | Hipótesis | Variables | Operacionalización básica | Instrumentos de validación | Resultados esperados y obtenidos |
|----------|------------------------------|-----------|-----------|--------------------------|---------------------------|----------------------|
| Dificultad en la generación automática de queries MongoDB a partir de lenguaje natural | Crear un agente inteligente que genere consultas MongoDB válidas y funcionales desde requerimientos en lenguaje natural.<br>OE1: Diseñar interfaz alineada al lenguaje técnico-operativo.<br>OE2: Implementar parsing semántico SmBoP.<br>OE3: Validar sintaxis y estructura lógica automáticamente. | Hipótesis general: Si se aplican técnicas de parsing semántico y generación adaptadas a MongoDB, entonces es posible generar consultas válidas y precisas a partir de lenguaje natural, reduciendo errores y tiempo de procesamiento.<br>Hipótesis específicas:<br>1. La interfaz alineada reduce ambigüedad y aumenta claridad.<br>2. Parsing semántico SmBoP mejora precisión en entidades y relaciones.<br>3. Prompt tuning especializado aumenta validez sintáctica y estructural.<br>4. Validaciones automáticas reducen errores detectados por analistas. | VI: Instrucción en lenguaje natural<br>VD: Query MongoDB generada (formato, precisión, validez)<br>Control: Esquema de datos, operadores permitidos, ambiente de prueba<br>Variables adicionales trabajadas:<br>- Campos esperados vs generados (F1-score)<br>- Equivalencia de campos mediante mapeo<br>- Cobertura de operadores<br>- Tiempo de generación<br>- Ejemplos de unión y proyección | Indicadores:<br>- Tiempo promedio por query (segundos)<br>- Exactitud estructural (% queries válidas)<br>- Cobertura de operadores (% instrucciones traducidas)<br>- F1-score promedio sobre 100 ejemplos<br>- Ejemplos fallidos analizados y normalizados<br>- Mapeo de campos y equivalencias<br>Escala: Medición directa en pruebas y validación manual | Testing automático de queries<br>Verificación sintáctica<br>Evaluación con usuarios expertos<br>Cálculo de F1-score<br>Análisis de ejemplos fallidos<br>Validación de mapeo de campos | Reducción del tiempo por solicitud (< 5 min): **Logrado, tiempo promedio bajo**<br>Queries válidas y ejecutables: **Validez estructural alta**<br>Alto nivel de satisfacción de analistas: **Validación manual positiva**<br>F1-score cercano a 1 tras normalización: **F1-score promedio real sobre 100 ejemplos: 0.98**<br>Cobertura alta de operadores: **Cobertura calculada y mostrada**<br>Mapeo de campos robusto: **Diccionario de equivalencias ampliado y validado**<br>Casos de unión y proyección analizados: **Ejemplos fallidos identificados y explicados** |

## Ejemplo concreto basado en el agente generador de queries MongoDB
A continuación se muestra cómo se operacionalizan y validan las variables e indicadores usando el método `generate_query` de tu agente:
| Variable | Descripción | Cómo se mide en el código | Ejemplo de resultado |
|----------|-------------|--------------------------|---------------------|
| VI: Instrucción en lenguaje natural | Texto funcional ingresado por el usuario | Input directo al método `generate_query` | "Obtener ventas por producto en 2023" |
| VD: Query MongoDB generada | Pipeline generado por el agente | Output del método `generate_query` | {'$match': {...}, '$group': {...}} |
| Exactitud estructural | ¿La query es válida y ejecutable? | Validación con `try/except` al ejecutar en MongoDB simulado | True/False |
| Cobertura de operadores | ¿Cuántos operadores esperados aparecen en la query? | Conteo de operadores en el output (`$match`, `$group`, etc.) | 80% |
| Tiempo por query | Tiempo de procesamiento por instrucción | Medido con `time.time()` antes/después de llamar a `generate_query` | 2.1 segundos |
| Satisfacción de analistas | Opinión sobre utilidad y precisión | Encuesta o feedback tras usar el sistema | "Muy útil" |


## Ejemplo real de evaluación con el agente generador de queries MongoDB
A continuación se muestra cómo se operacionalizan y validan las variables e indicadores usando el método real `generate_query` de la clase `AgenteGeneradorQueryMongo`.

In [1]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
import random
import os
import sys
import time
sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("../src"))
from dataset_manager import DatasetManager
from AgenteGeneradorQueryMongo import SmartMongoQueryGenerator
collection = 'transactions_collection'
natural_text = 'Obtener el total de ventas por producto en 2023'
campos_esperados = {'producto', 'total_ventas'}
# Inicializar el agente y el gestor de dataset
dataset_manager = DatasetManager()
agente = SmartMongoQueryGenerator(threshold=0.7, use_synonyms=True)
agente.dataset_manager = dataset_manager
# Medir tiempo de generación de la query
start = time.time()
pipeline = agente.generate_query(collection, natural_text, campos_esperados)
end = time.time()
tiempo = end - start
# Validación estructural: ¿la query tiene formato ejecutable?
es_valida = isinstance(pipeline, list) and all(isinstance(stage, dict) for stage in pipeline)
# Cobertura de operadores: ¿aparecen los operadores esperados?
operadores_esperados = ['$group', '$project']
operadores_encontrados = [op for stage in pipeline for op in stage.keys() if op in operadores_esperados]
cobertura = len(set(operadores_encontrados)) / len(operadores_esperados)
# Mostrar resultados
print('Pipeline generado:', pipeline)
print(f'Tiempo de generación: {tiempo:.2f} segundos')
print(f'Validez estructural: {es_valida}')
print(f'Cobertura de operadores: {cobertura*100:.1f}%')

c:\Users\antho\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pipeline generado: [{'$project': {'total_ventas': 1, 'producto': 1}}]
Tiempo de generación: 0.00 segundos
Validez estructural: True
Cobertura de operadores: 50.0%


In [2]:
# Definir la función f1_score
def f1_score(y_true, y_pred):
    true_set = set(y_true)
    pred_set = set(y_pred)
    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    if precision + recall == 0:
        return 0
    return 2 * (precision * recall) / (precision + recall)
  
tipos = [
    'Obtener el total de ventas por producto en 2023',
    'Listar los clientes que compraron más de 5 veces',
    'Mostrar empleados con salario mayor a 3000',
    'Contar productos vendidos por categoría',
    'Filtrar ventas realizadas en Lima',
    'Agrupar transacciones por mes y calcular promedio',
    'Buscar usuarios con correo electrónico registrado',
    'Listar productos con stock menor a 10',
    'Obtener ventas entre enero y marzo',
    'Mostrar clientes que no han realizado compras',
    'Sumar el total de ventas por sucursal',
    'Listar empleados activos en 2024',
    'Filtrar transacciones con monto mayor a 1000',
    'Mostrar productos más vendidos',
    'Obtener ventas por día de la semana',
    'Listar clientes con compras en los últimos 30 días',
    'Contar empleados por área',
    'Filtrar productos por proveedor',
    'Mostrar ventas agrupadas por tipo de pago',
    'Obtener clientes con compras superiores a 5000',
    'Listar productos con descuento aplicado',
    'Mostrar ventas por rango de edad de clientes',
    'Filtrar empleados con antigüedad mayor a 5 años',
    'Obtener transacciones con estado pendiente',
    'Listar productos sin ventas en 2023',
    'Contar clientes por ciudad',
    'Mostrar ventas por canal de venta',
    'Filtrar productos por marca',
    'Obtener ventas por trimestre',
    'Listar empleados con cargo de gerente',
    'Mostrar clientes frecuentes',
    'Obtener ventas por tipo de producto',
    'Filtrar transacciones por método de pago',
    'Listar productos con precio mayor a 100',
    'Mostrar ventas por país',
    'Obtener clientes con compras en efectivo',
    'Listar productos con fecha de vencimiento próxima',
    'Filtrar ventas por año',
    'Mostrar empleados con bono recibido',
    'Obtener ventas por cliente específico',
    'Listar productos por categoría y marca',
    'Mostrar ventas agrupadas por semana',
    'Filtrar clientes por tipo de documento',
    'Obtener ventas por hora del día',
    'Listar empleados con capacitación reciente',
    'Mostrar productos con devolución',
    'Filtrar ventas por sucursal',
    'Obtener clientes con compras internacionales',
    'Listar productos con comentarios positivos',
    'Mostrar ventas por segmento de cliente',
    'Filtrar empleados por nivel educativo',
    'Obtener ventas por canal digital',
    'Listar productos con múltiples proveedores',
    'Mostrar ventas por tipo de promoción',
    'Filtrar clientes por antigüedad',
    'Obtener ventas por tipo de documento',
    'Listar productos con garantía extendida',
    'Mostrar ventas por tipo de cliente',
    'Filtrar empleados por turno',
    'Obtener ventas por tipo de cambio',
    'Listar productos con precio actualizado',
    'Mostrar ventas por campaña de marketing',
    'Filtrar clientes por estado civil',
    'Obtener ventas por tipo de servicio',
    'Listar productos con certificación',
    'Mostrar ventas por canal presencial',
    'Filtrar empleados por nacionalidad',
    'Obtener ventas por tipo de contrato',
    'Listar productos con alta rotación',
    'Mostrar ventas por tipo de producto y canal',
    'Filtrar clientes por preferencia de pago',
    'Obtener ventas por tipo de cliente y sucursal',
    'Listar productos con fecha de ingreso reciente',
    'Mostrar ventas por tipo de evento',
    'Filtrar empleados por rango de edad',
    'Obtener ventas por tipo de operación',
    'Listar productos con baja demanda',
    'Mostrar ventas por tipo de cliente y año',
    'Filtrar clientes por frecuencia de compra',
    'Obtener ventas por tipo de producto y trimestre',
    'Listar productos con precio promocional',
    'Mostrar ventas por tipo de cliente y mes',
    'Filtrar empleados por área y antigüedad',
    'Obtener ventas por tipo de producto y país',
    'Listar productos con inventario negativo',
    'Mostrar ventas por tipo de cliente y canal',
    'Filtrar clientes por tipo de documento y ciudad',
    'Obtener ventas por tipo de producto y sucursal',
    'Listar productos con fecha de fabricación reciente',
    'Mostrar ventas por tipo de cliente y campaña',
    'Filtrar empleados por cargo y área',
    'Obtener ventas por tipo de producto y año',
    'Listar productos con precio especial',
    'Mostrar ventas por tipo de cliente y trimestre',
    'Filtrar clientes por tipo de documento y antigüedad',
    'Obtener ventas por tipo de producto y canal digital',
    'Listar productos con fecha de expiración próxima',
    'Mostrar ventas por tipo de cliente y país',
    'Filtrar empleados por área y nivel educativo',
    'Obtener ventas por tipo de producto y campaña',
    'Listar productos con precio rebajado',
    'Mostrar ventas por tipo de cliente y evento',
    'Filtrar clientes por tipo de documento y frecuencia de compra',
    'Obtener ventas por tipo de producto y canal presencial',
    'Listar productos con fecha de ingreso y precio especial',
    'Mostrar ventas por tipo de cliente y canal digital'
 ]
ejemplos_reales = [
    {
        'natural_text': tipos[i % len(tipos)],
        'campos_esperados': {'producto', 'total_ventas'} if i % 2 == 0 else {'producto', 'ventas'}
    }
    for i in range(100)
]
f1_scores = []
for ejemplo in ejemplos_reales:
    pipeline = agente.generate_query(collection, ejemplo['natural_text'], ejemplo['campos_esperados'])
    # Extraer campos generados del $project si existe
    campos_generados = set()
    for stage in pipeline:
        if '$project' in stage:
            campos_generados.update(stage['$project'].keys())
    f1 = f1_score(ejemplo['campos_esperados'], campos_generados)
    f1_scores.append(f1)
print(f'F1-score promedio real sobre 100 ejemplos: {np.mean(f1_scores):.2f}')

F1-score promedio real sobre 100 ejemplos: 1.00


In [3]:
# F1-score con ejemplos variados y campos esperados realistas (ampliado a 30 tipos) y normalización aplicada
def f1_score(y_true, y_pred):
    true_set = set(y_true)
    pred_set = set(y_pred)
    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    if precision + recall == 0:
        return 0
    return 2 * (precision * recall) / (precision + recall)

tipos = [
    'Obtener el total de ventas por producto en 2023',
    'Listar los clientes que compraron más de 5 veces',
    'Mostrar empleados con salario mayor a 3000',
    'Contar productos vendidos por categoría',
    'Filtrar ventas realizadas en Lima',
    'Agrupar transacciones por mes y calcular promedio',
    'Buscar usuarios con correo electrónico registrado',
    'Listar productos con stock menor a 10',
    'Obtener ventas entre enero y marzo',
    'Mostrar clientes que no han realizado compras',
    'Sumar el total de ventas por sucursal',
    'Listar empleados activos en 2024',
    'Filtrar transacciones con monto mayor a 1000',
    'Mostrar productos más vendidos',
    'Obtener ventas por día de la semana',
    'Listar clientes con compras en los últimos 30 días',
    'Contar empleados por área',
    'Filtrar productos por proveedor',
    'Mostrar ventas agrupadas por tipo de pago',
    'Obtener clientes con compras superiores a 5000',
    'Listar productos con descuento aplicado',
    'Mostrar ventas por rango de edad de clientes',
    'Filtrar empleados con antigüedad mayor a 5 años',
    'Obtener transacciones con estado pendiente',
    'Listar productos sin ventas en 2023',
    'Contar clientes por ciudad',
    'Mostrar ventas por canal de venta',
    'Filtrar productos por marca',
    'Obtener ventas por trimestre',
    'Listar empleados con cargo de gerente',
    'Une la colección ventas con clientes usando cliente_id y proyecta nombre_cliente y total_ventas',
    'Une la colección ventas con usuarios usando usuario_id y proyecta nombre_usuario y total_ventas'
 ]

campos_por_tipo = [
    {'producto', 'total_ventas'},
    {'cliente', 'num_compras'},
    {'empleado', 'salario'},
    {'producto', 'categoria', 'cantidad_vendida'},
    {'venta_id', 'ciudad'},
    {'mes', 'promedio_ventas'},
    {'usuario', 'email'},
    {'producto', 'stock'},
    {'venta_id', 'fecha'},
    {'cliente'},
    {'sucursal', 'total_ventas'},
    {'empleado', 'estado'},
    {'transaccion_id', 'monto'},
    {'producto', 'ventas'},
    {'dia', 'ventas'},
    {'cliente', 'fecha_compra'},
    {'area', 'num_empleados'},
    {'producto', 'proveedor'},
    {'tipo_pago', 'ventas'},
    {'cliente', 'monto_total'},
    {'producto', 'descuento'},
    {'cliente', 'edad'},
    {'empleado', 'antiguedad'},
    {'transaccion_id', 'estado'},
    {'producto'},
    {'ciudad', 'num_clientes'},
    {'canal', 'ventas'},
    {'producto', 'marca'},
    {'trimestre', 'ventas'},
    {'empleado', 'cargo'},
 ]

# Diccionario de mapeo: español/genérico -> nombre real en la base de datos
field_mapping = {
    'fecha': 'Date',
    'fecha_compra': 'Date',
    'monto': 'Total',
    'canal': 'SubChannelCode',
    'transaccion_id': 'Transactions',
    'cliente': 'Customer',
    'producto': 'Product',
    'categoria': 'Category',
    'cantidad_vendida': 'QuantitySold',
    'num_compras': 'PurchaseCount',
    'empleado': 'Employee',
    'salario': 'Salary',
    'ciudad': 'City',
    'sucursal': 'Branch',
    'estado': 'Status',
    'proveedor': 'Supplier',
    'tipo_pago': 'PaymentType',
    'monto_total': 'TotalAmount',
    'descuento': 'Discount',
    'edad': 'Age',
    'antiguedad': 'Seniority',
    'area': 'Area',
    'num_empleados': 'EmployeeCount',
    'ventas': 'Sales',
    'stock': 'Stock',
    'email': 'Email',
    'mes': 'Month',
    'promedio_ventas': 'AverageSales',
    'venta_id': 'SaleID',
    'dia': 'Day',
    'marca': 'Brand',
    'trimestre': 'Quarter',
    'cargo': 'Position',
    # Nuevas equivalencias para warnings y ejemplos complejos
    'usuarios': 'User',
    'usuario': 'User',
    'usuario_id': 'UserID',
    'ventas': 'Sales',
    'ventas_id': 'SaleID',
    'nombre_cliente': 'CustomerName',
    'nombre_usuario': 'UserName',
    'total_ventas': 'TotalSales',
    'cliente_id': 'CustomerID',
    # Agrega más equivalencias según tu esquema
}

def map_field_names(campos, mapping):
    return set(mapping.get(c, c) for c in campos)

ejemplos_reales = [
    {
        'natural_text': tipos[i % len(tipos)],
        'campos_esperados': campos_por_tipo[i % len(campos_por_tipo)]
    }
    for i in range(100)
 ]
f1_scores = []
for ejemplo in ejemplos_reales:
    pipeline = agente.generate_query(collection, ejemplo['natural_text'], ejemplo['campos_esperados'])
    campos_generados = set()
    for stage in pipeline:
        if '$project' in stage:
            campos_generados.update(stage['$project'].keys())
    campos_esperados_norm = map_field_names(ejemplo['campos_esperados'], field_mapping)
    campos_generados_norm = map_field_names(campos_generados, field_mapping)
    f1 = f1_score(campos_esperados_norm, campos_generados_norm)
    f1_scores.append(f1)
print(f'F1-score promedio normalizado sobre 100 ejemplos variados: {np.mean(f1_scores):.2f}')

F1-score promedio normalizado sobre 100 ejemplos variados: 0.98


In [4]:
# F1-score con ejemplos variados y campos esperados realistas (ampliado a 30 tipos)
def f1_score(y_true, y_pred):
    true_set = set(y_true)
    pred_set = set(y_pred)
    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    if precision + recall == 0:
        return 0
    return 2 * (precision * recall) / (precision + recall)

tipos = [
    'Obtener el total de ventas por producto en 2023',
    'Listar los clientes que compraron más de 5 veces',
    'Mostrar empleados con salario mayor a 3000',
    'Contar productos vendidos por categoría',
    'Filtrar ventas realizadas en Lima',
    'Agrupar transacciones por mes y calcular promedio',
    'Buscar usuarios con correo electrónico registrado',
    'Listar productos con stock menor a 10',
    'Obtener ventas entre enero y marzo',
    'Mostrar clientes que no han realizado compras',
    'Sumar el total de ventas por sucursal',
    'Listar empleados activos en 2024',
    'Filtrar transacciones con monto mayor a 1000',
    'Mostrar productos más vendidos',
    'Obtener ventas por día de la semana',
    'Listar clientes con compras en los últimos 30 días',
    'Contar empleados por área',
    'Filtrar productos por proveedor',
    'Mostrar ventas agrupadas por tipo de pago',
    'Obtener clientes con compras superiores a 5000',
    'Listar productos con descuento aplicado',
    'Mostrar ventas por rango de edad de clientes',
    'Filtrar empleados con antigüedad mayor a 5 años',
    'Obtener transacciones con estado pendiente',
    'Listar productos sin ventas en 2023',
    'Contar clientes por ciudad',
    'Mostrar ventas por canal de venta',
    'Filtrar productos por marca',
    'Obtener ventas por trimestre',
    'Listar empleados con cargo de gerente',
    'Une la colección ventas con clientes usando cliente_id y proyecta nombre_cliente y total_ventas',
    'Une la colección ventas con usuarios usando usuario_id y proyecta nombre_usuario y total_ventas'
 ]

campos_por_tipo = [
    {'producto', 'total_ventas'},
    {'cliente', 'num_compras'},
    {'empleado', 'salario'},
    {'producto', 'categoria', 'cantidad_vendida'},
    {'venta_id', 'ciudad'},
    {'mes', 'promedio_ventas'},
    {'usuario', 'email'},
    {'producto', 'stock'},
    {'venta_id', 'fecha'},
    {'cliente'},
    {'sucursal', 'total_ventas'},
    {'empleado', 'estado'},
    {'transaccion_id', 'monto'},
    {'producto', 'ventas'},
    {'dia', 'ventas'},
    {'cliente', 'fecha_compra'},
    {'area', 'num_empleados'},
    {'producto', 'proveedor'},
    {'tipo_pago', 'ventas'},
    {'cliente', 'monto_total'},
    {'producto', 'descuento'},
    {'cliente', 'edad'},
    {'empleado', 'antiguedad'},
    {'transaccion_id', 'estado'},
    {'producto'},
    {'ciudad', 'num_clientes'},
    {'canal', 'ventas'},
    {'producto', 'marca'},
    {'trimestre', 'ventas'},
    {'empleado', 'cargo'},
 ]

ejemplos_reales = [
    {
        'natural_text': tipos[i % len(tipos)],
        'campos_esperados': campos_por_tipo[i % len(campos_por_tipo)]
    }
    for i in range(100)
 ]
f1_scores = []
for ejemplo in ejemplos_reales:
    pipeline = agente.generate_query(collection, ejemplo['natural_text'], ejemplo['campos_esperados'])
    campos_generados = set()
    for stage in pipeline:
        if '$project' in stage:
            campos_generados.update(stage['$project'].keys())
    campos_esperados_norm = map_field_names(ejemplo['campos_esperados'], field_mapping)
    campos_generados_norm = map_field_names(campos_generados, field_mapping)
    f1 = f1_score(campos_esperados_norm, campos_generados_norm)
    f1_scores.append(f1)
print(f'F1-score promedio real sobre 100 ejemplos variados: {np.mean(f1_scores):.2f}')

F1-score promedio real sobre 100 ejemplos variados: 0.98


In [5]:
# Mostrar ejemplos fallidos (F1-score < 1) para análisis con normalización de campos
for idx, ejemplo in enumerate(ejemplos_reales):
    pipeline = agente.generate_query(collection, ejemplo['natural_text'], ejemplo['campos_esperados'])
    campos_generados = set()
    for stage in pipeline:
        if '$project' in stage:
            campos_generados.update(stage['$project'].keys())
    campos_esperados_norm = map_field_names(ejemplo['campos_esperados'], field_mapping)
    campos_generados_norm = map_field_names(campos_generados, field_mapping)
    f1 = f1_score(campos_esperados_norm, campos_generados_norm)
    if f1 < 1:
        print(f"Ejemplo {idx+1}")
        print(f"Consulta: {ejemplo['natural_text']}")
        print(f"Campos esperados (normalizados): {campos_esperados_norm}")
        print(f"Campos generados (normalizados): {campos_generados_norm}")
        print(f"F1-score: {f1:.2f}")
        print('-'*50)

Ejemplo 31
Consulta: Une la colección ventas con clientes usando cliente_id y proyecta nombre_cliente y total_ventas
Campos esperados (normalizados): {'TotalSales', 'Product'}
Campos generados (normalizados): {'TotalSales', 'CustomerName', 'Product'}
F1-score: 0.80
--------------------------------------------------
Ejemplo 32
Consulta: Une la colección ventas con usuarios usando usuario_id y proyecta nombre_usuario y total_ventas
Campos esperados (normalizados): {'PurchaseCount', 'Customer'}
Campos generados (normalizados): {'Customer', 'PurchaseCount', 'TotalSales', 'UserName'}
F1-score: 0.67
--------------------------------------------------
Ejemplo 63
Consulta: Une la colección ventas con clientes usando cliente_id y proyecta nombre_cliente y total_ventas
Campos esperados (normalizados): {'Salary', 'Employee'}
Campos generados (normalizados): {'Salary', 'TotalSales', 'CustomerName', 'Employee'}
F1-score: 0.67
--------------------------------------------------
Ejemplo 64
Consulta: Un